### New countries in the world


| Country | ISO3 | Independence | Parent nation | ISO3- parent nation| Divorce/Unification|
| --- | --- | --- | --- | --- | --- |
| Bosnia and Herzegovina | BIH | 1992 | Yugoslavia | YUG | Separation |
| Eritrea | ERI | 1993 | Ethiopia | ETH | Separation | 
| Korea, Democratic People's Republic of (North Korea) | PRK | 1948 | Japan | JPN | Separation | 
| Yemen, Republic of | YEM | 1990 | Yemen Arab Republic (North Yemen), People's Democratic Republic of Yemen (South Yemen) | YAR, YMD | Unification | 
| Somalia, Fed. Rep. | SOM | 1960 | Trust Territory of Somaliland (Italy), British Somaliland (UK) | IT-SOM, GB-SOM | Unification |
| Namibia | NAM | 1990 | South Africa | ZAF | Separation |
| Croatia | HRV | 1991 | Yugoslavia | YUG | Separation |
| Slovenia | SVN | 1991 | Yugoslavia | YUG | Separation |
| North Macedonia | MKD | 1991 | Yugoslavia | YUG | Separation |
| Czech Republic (Czechia) | CZE | 1993 | Czechoslovakia | CSK | Separation |
| Slovakia | SVK | 1993 | Czechoslovakia | CSK | Separation |
| Palau | PLW | 1994 | United States (UN Trust Territory) | USA | Separation |
| Timor-Leste | TLS | 2002 | Indonesia | IDN | Separation |
| Montenegro | MNE | 2006 | Serbia and Montenegro | SCG | Separation |
| Serbia | SRB | 2006 | Serbia and Montenegro | SCG | Separation |
| South Sudan | SSD | 2011 | Sudan | SDN | Separation |
| Kosovo (Partial recognition) | XKX / RKS | 2008 | Serbia | SRB | Separation |
| Curaçao | CUW | 2010 | Netherlands Antilles | ANT | Separation |
| Sint Maarten (Dutch part) | SXM | 2010 | Netherlands Antilles | ANT | Separation |
| Bonaire, Sint Eustatius and Saba | BES | 2010 | Netherlands Antilles | ANT | Separation |
| Netherlands Antilles (Historical) | ANT | 1954 | Netherlands, The | NLD | Separation |


##### origin_stock = people born in Afghanistan living abroad (emigrant stock)
##### destination_stock = foreign-born people living in Afghanistan (immigrant stock)

## Inspecting 

In [1]:
import pandas as pd

In [2]:
mig = pd.read_csv("../data/cleaned/panel_migration.csv")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_10352\3919046788.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  mig = pd.read_csv("../data/cleaned/panel_migration.csv")


In [3]:
mig = mig.sort_values('year')
# creating pair tuple
mig['pair'] = list(zip(mig['origin'], mig['destination']))
# storing pair sets for each year
pairs_by_year = {y: set(df['pair']) for y, df in mig.groupby('year')}
years = sorted(mig['year'].unique())
results = []
for i, y in enumerate(years):
    current_pairs = pairs_by_year[y]
    if i == 0:
        appeared = current_pairs
        disappeared = set()
    else:
        prev_pairs = pairs_by_year[years[i-1]]
        # appeared in current year
        appeared = current_pairs - prev_pairs
        # disappeared from previous year
        disappeared = prev_pairs - current_pairs
    results.append({
        'year': y,
        # counts
        'n_country_pairs': len(current_pairs),
        'n_new_pairs': len(appeared),
        'n_disappeared_pairs': len(disappeared),
        # pair lists
        'new_pairs_list': sorted(list(appeared)),
        'disappeared_pairs_list': sorted(list(disappeared)),
        # country counts
        'n_origin_countries': mig.loc[mig['year'] == y, 'origin'].nunique(),
        'n_destination_countries': mig.loc[mig['year'] == y, 'destination'].nunique()
    })

summary_df1 = pd.DataFrame(results)

In [4]:
summary_df1.head()

,year,n_country_pairs,n_new_pairs,n_disappeared_pairs,new_pairs_list,disappeared_pairs_list,n_origin_countries,n_destination_countries
0,1990,43890,43890,0,"[(Afghanistan, Albania), (Afghanistan, Algeria...",[],210,210
1,1991,43890,0,0,[],[],210,210
2,1992,43890,0,0,[],[],210,210
3,1993,43890,0,0,[],[],210,210
4,1994,43890,0,0,[],[],210,210


In [5]:
summary_df1.to_csv("../data/cleaned/summary_migration_data.csv", encoding = "utf-8")

In [6]:
mig['year'].unique()

array([1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000,
       2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011,
       2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020], dtype=int64)

In [7]:
print(mig['destination'].nunique())
print(mig['origin'].nunique())

210
210


## Filtering the data

#### Step 1: Counting the sum of stock for each country when it is origin and again when it destination

In [8]:
# Counting lowest 5% of emigrant stock for origin countries
origin_year = (mig.groupby(['origin', 'year'])['stock'].sum().reset_index(name='tot_im'))
origin_max = (origin_year.groupby('origin')['tot_im'].max().rename('max_tot_im_orgn'))
origin_threshold = origin_max.quantile(0.05)
origin_flag = pd.DataFrame({'max_tot_im_orgn': origin_max,'low_5prc_orgn': origin_max < origin_threshold})

In [9]:
# Counting lowest 5% of immigrant stock for destination countries
destination_year = (mig.groupby(['destination', 'year'])['stock'].sum().reset_index(name='tot_im'))
destination_max = (destination_year.groupby('destination')['tot_im'].max().rename('max_tot_im_dstn'))
destination_threshold = destination_max.quantile(0.05)
destination_flag = pd.DataFrame({'max_tot_im_dstn': destination_max,'low_5prc_dstn': destination_max < destination_threshold})

In [10]:
origin_flag[origin_flag['low_5prc_orgn'] == True]

,max_tot_im_orgn,low_5prc_orgn
origin,,
Cayman Islands,5745.0,True
Kiribati,10620.0,True
Liechtenstein,3851.0,True
Maldives,4614.0,True
Nauru,2700.0,True
San Marino,4537.0,True
Sint Maarten (Dutch part),3303.0,True
Solomon Islands,6254.0,True
Turks and Caicos Islands,4145.0,True


In [11]:
destination_flag[destination_flag['low_5prc_dstn'] == True]

,max_tot_im_dstn,low_5prc_dstn
destination,,
Kiribati,4154.0,True
Kosovo,0.0,True
Marshall Islands,3752.0,True
Micronesia; Fed. Sts.,2265.0,True
Nauru,3997.0,True
Solomon Islands,5627.0,True
St. Kitts and Nevis,5508.0,True
St. Vincent and the Grenadines,4847.0,True
Tonga,2749.0,True


In [12]:
destination_flag.index.name = 'country'
origin_flag.index.name = 'country'

In [13]:
country_filter = (origin_flag.merge(destination_flag, left_index=True, right_index=True, how='outer').fillna(False))

In [14]:
country_filter[(country_filter['low_5prc_orgn'] == True) & (country_filter['low_5prc_dstn'] == True)]

,max_tot_im_orgn,low_5prc_orgn,max_tot_im_dstn,low_5prc_dstn
country,,,,
Kiribati,10620.0,True,4154.0,True
Nauru,2700.0,True,3997.0,True
Solomon Islands,6254.0,True,5627.0,True
Tuvalu,3589.0,True,489.0,True
Vanuatu,8505.0,True,2884.0,True


#### Step 2: Dropping countries in bottom 5% on either sides

In [15]:
drop_countries = country_filter[country_filter['low_5prc_orgn'] & country_filter['low_5prc_dstn']].index

In [16]:
print(len(drop_countries)) # 5 countries drop in this step

5


In [17]:
print(drop_countries)

Index(['Kiribati', 'Nauru', 'Solomon Islands', 'Tuvalu', 'Vanuatu'], dtype='object', name='country')


In [18]:
mig_filtered = mig[~mig['origin'].isin(drop_countries) & ~mig['destination'].isin(drop_countries)]

In [19]:
print(mig['destination'].nunique())
print(mig['origin'].nunique())
print(mig.groupby(['origin', 'destination']).ngroups)
print(mig['year'].nunique())

210
210
43890
31


In [20]:
print(mig_filtered['destination'].nunique())
print(mig_filtered['origin'].nunique())
print(mig_filtered.groupby(['origin', 'destination']).ngroups)
print(mig_filtered['year'].nunique())

205
205
41820
31


In [21]:
mig_filtered.columns

Index(['origin', 'destination', 'year', 'stock', 'flow', 'iso_o', 'region',
       'region_d', 'GDP_pc', 'GDP_pc_d', 'mortality', 'mortality_d',
       'primary_enroll', 'secondary_enroll', 'secondary_enroll_d',
       'tertiary_enroll', 'tertiary_enroll_d', 'maternal_death_risk',
       'maternal_death_risk_d', 'mmr', 'mmr_d', 'primary_enroll_d',
       'literacy_rate', 'literacy_rate_d', 'gini', 'gini_d',
       'electoral_dem_index', 'electoral_dem_index_d', 'nourishment',
       'nourishment_d', 'government_effec_index', 'government_effec_index_d',
       'political_corruption_index', 'political_corruption_index_d', 'gii',
       'gii_d', 'lfp_f', 'lfp_f_d', 'parliament_seat_f', 'parliament_seat_f_d',
       'hdi', 'hdi_d', 'life_exp', 'life_exp_d', 'mean_schooling',
       'mean_schooling_d', 'coef_human_inequality', 'coef_human_inequality_d',
       'inequality_edu', 'inequality_edu_d', 'inequality_income',
       'inequality_income_d', 'inequality_le', 'inequality_le_d', 'idealp

In [22]:
set(mig_filtered[mig_filtered['year'] == 1990]['destination'].unique()) - set(mig_filtered[mig_filtered['year'] == 1990]['origin'].unique())

set()

In [23]:
set(mig_filtered[mig_filtered['year'] == 2009]['destination'].unique()) - set(mig_filtered[mig_filtered['year'] == 2009]['origin'].unique())

set()

In [24]:
set(mig_filtered[mig_filtered['year'] == 2010]['destination'].unique()) - set(mig_filtered[mig_filtered['year'] == 2010]['origin'].unique())

set()

In [25]:
set(mig_filtered[mig_filtered['year'] == 2019]['destination'].unique()) - set(mig_filtered[mig_filtered['year'] == 2019]['origin'].unique())

set()

In [26]:
set(mig_filtered[mig_filtered['year'] == 2020]['destination'].unique()) - set(mig_filtered[mig_filtered['year'] == 2020]['origin'].unique())

set()

In [27]:
assert set(mig_filtered[mig_filtered['year'] == 1990]['origin'].unique()) == set(mig_filtered[mig_filtered['year'] == 1990]['destination'].unique())

In [28]:
assert set(mig_filtered[mig_filtered['year'] == 1990]['origin'].unique()) == set(mig_filtered[mig_filtered['year'] == 1990]['destination'].unique()) == set(mig_filtered[mig_filtered['year'] == 1990]['origin'].unique()) == set(mig_filtered[mig_filtered['year'] == 1990]['destination'].unique()) 

In [29]:
attributes = pd.read_csv("../data/cleaned/attributes.csv")
edgelist_flow = pd.read_csv("../data/cleaned/edgelist_flow.csv")
edgelist_stock = pd.read_csv("../data/cleaned/edgelist_stock.csv")

In [30]:
attributes = attributes[~attributes['country'].isin(drop_countries)]

In [31]:
attributes['country'].nunique()

205

In [32]:
edgelist_flow = edgelist_flow[~edgelist_flow['origin'].isin(drop_countries)]
edgelist_flow = edgelist_flow[~edgelist_flow['destination'].isin(drop_countries)]
edgelist_stock = edgelist_stock[~edgelist_stock['origin'].isin(drop_countries)]
edgelist_stock = edgelist_stock[~edgelist_stock['destination'].isin(drop_countries)]

In [33]:
print(edgelist_stock['origin'].nunique())
print(edgelist_stock['destination'].nunique())

205
205


In [34]:
mig_filtered.to_csv('../data/cleaned/panel_migration.csv', encoding = 'utf-8', index=False)
attributes.to_csv('../data/cleaned/attributes.csv', encoding = 'utf-8', index=False)
edgelist_flow.to_csv('../data/cleaned/edgelist_flow.csv', encoding = 'utf-8', index=False)
edgelist_stock.to_csv('../data/cleaned/edgelist_stock.csv', encoding = 'utf-8', index=False)